# RetiZero runs

Everything for RetiZero in one queue: the replay of the saved checkpoint, the training runs (original split, clean split, CLAHE), timing, and Messidor scoring. It works the same way as `finetuning/Qwen3/notebooks/03_final_runs.ipynb`.

1. Run **1. Setup**, then **2. What is left**. Both only read files.
2. Set `TRAINING_MODE = True` in Setup and rerun it when you want to start GPU jobs.
3. Run **4. Pilot** first. It takes about three minutes.
4. Run **5. Full queue** and leave it. It works through every job whose inputs are present, records a failed job and carries on. Rerunning the cell skips finished jobs and resumes interrupted training from its last epoch.

Training needs the official pretrained `RetiZero.pth`, which is not on this PC yet. Until it is, only the replay of the saved checkpoint runs, and the pilot trains from random weights to test the code. Once the file is in place, paste its SHA-256 into Setup, repeat the pilot with a new `pilot_tag`, then start the full queue.

Results are written to `D:\Afterimage_Experiments\retizero_runs\<tag>`. RetiZero jobs take the same GPU lock as Qwen jobs, so the two queues cannot overlap.

[Plan and reasoning](../REPRODUCTION_PLAN.md) · [Folder guide](../README.md) · [Deliverables checklist](../../../docs/paper/DELIVERABLES_CHECKLIST.md)

## 1. Setup

In [ ]:
from pathlib import Path
import importlib
import sys

if Path(sys.prefix).name.casefold() != "afterimage_env" or sys.version_info[:2] != (3, 11):
    raise RuntimeError(f"Select the Python 3.11 afterimage_env kernel. Current: {sys.executable}")

sys.dont_write_bytecode = True
RETIZERO_DIR = Path(r"D:\Afterimage_Reproduction\finetuning\RetiZero")
HELPER_DIR = RETIZERO_DIR / "paper_revision"  # retizero_reproduction.py stays in RETIZERO_DIR
if str(HELPER_DIR) in sys.path:
    sys.path.remove(str(HELPER_DIR))
sys.path.insert(0, str(HELPER_DIR))
importlib.invalidate_caches()

def load_helper(name):
    module = importlib.reload(sys.modules[name]) if name in sys.modules else importlib.import_module(name)
    if Path(module.__file__).resolve() != (HELPER_DIR / f"{name}.py").resolve():
        raise ImportError(f"Expected {name} from {HELPER_DIR}, got {module.__file__}")
    return module

runs = load_helper("retizero_runs")

TRAINING_MODE = False  # Change to True and rerun this cell to enable GPU jobs.

SETTINGS = runs.default_settings()
SETTINGS["tag"] = "retizero_01"     # one folder per campaign; keep it to resume or reuse results
SETTINGS["pilot_tag"] = "pilot_01"  # change it to repeat the pilot from scratch

# The official pretrained RetiZero checkpoint. Every training run needs it; the replay does not.
# Download link: RetiZero/RetiZero/README.md. Section 2 prints the SHA-256 to paste here.
SETTINGS["pretrained"] = Path(r"D:\Afterimage_Experiments\assets\RetiZero.pth")
SETTINGS["pretrained_sha256"] = "7d06d29441c981898bbb56abdaae1eb97e205448f0a9bc2a68e2aa9431601ede"

# Stages. Turn one off to leave it out of the queue.
SETTINGS["stages"].update(
    replay=True,             # the saved app checkpoint on the original 367 and the clean 361 test images
    historical=True,         # original recipe on the original split, three seeds
    matched=True,            # the paper's comparison: clean split, same 361 test IDs as Qwen, three seeds
    clahe=True,              # S9: the same on a CLAHE build (waits until that dataset exists)
    benchmark=True,          # S14: time per image, memory and parameters for each matched run
    messidor_archived=True,  # archived Messidor at 224 pixels (waits until that build exists)
)
SETTINGS["seeds"] = [42, 43, 44]   # the Qwen trainer seeds
SETTINGS["effective_batch"] = 128  # images per optimizer step, as in the original notebook
SETTINGS["physical_batch"] = 32    # images per forward pass; lower it if the pilot reports memory pressure

def launch(job, *args, **kwargs):
    if not TRAINING_MODE:
        print("Set TRAINING_MODE = True and rerun Setup, then run this cell.")
        return
    return job(*args, **kwargs)

print("Setup OK. Results folder:", runs.root(SETTINGS))

## 2. What is left

Checks every input (checkpoints, datasets, GPU lock), then lists every job with its status and a time estimate. Reads saved files only. A job marked `waiting` has a missing input, named in brackets.

In [ ]:
checks = runs.checks(SETTINGS)

In [ ]:
jobs = runs.show_plan(SETTINGS)

## 3. Parameter counts (no GPU)

Counts the classifier's parameters directly from the saved checkpoint. The original notebook set every encoder weight trainable, so fine-tuning trains the whole model, not only the LoRA branches.

In [ ]:
counts = runs.parameter_counts(SETTINGS)

## 4. Pilot

Four small jobs: the saved checkpoint on 12 test images; a two-epoch training run on a balanced subset (32 images per grade); the same run stopped after its first epoch and resumed; and a scoring job on the pilot's own checkpoint. It then checks that the interrupted run matches the uninterrupted one, that training fits in GPU memory, and that the scoring worker reproduces the training worker's predictions. Check that the output ends with **Pilot passed** before starting the full queue.

In [ ]:
pilot_jobs = runs.show_plan(SETTINGS, pilot=True)

In [ ]:
launch(runs.run_queue, SETTINGS, pilot=True)

## 5. Full queue

Runs every job whose inputs are present, in order, and writes the summary at the end. Jobs with a missing input are skipped and stay `waiting`. If the queue is interrupted, run this cell again.

In [ ]:
launch(runs.run_queue, SETTINGS)

## 6. One stage at a time

Use these instead of the full queue to run or repeat a single stage.

In [ ]:
launch(runs.run_queue, SETTINGS, only={"replay"})

In [ ]:
launch(runs.run_queue, SETTINGS, only={"matched"})

In [ ]:
launch(runs.run_queue, SETTINGS, only={"historical"})

In [ ]:
launch(runs.run_queue, SETTINGS, only={"clahe"})

In [ ]:
launch(runs.run_queue, SETTINGS, only={"benchmark"})

In [ ]:
launch(runs.run_queue, SETTINGS, only={"messidor_archived"})

## 7. Results

Rebuilds `summary.csv` and `summary.md` from every finished job and shows the summary, including the comparison of the replay with the original notebook's report. Reads saved files only.

In [ ]:
rows = runs.summarize(SETTINGS)
summary = runs.root(SETTINGS) / "summary.md"
if summary.is_file():
    from IPython.display import Markdown, display
    display(Markdown(summary.read_text(encoding="utf-8")))

## 8. Not covered here

- **The CLAHE dataset and the Messidor build at 224 pixels.** The `clahe` and `messidor_archived` stages wait for them. They are step 9 of the plan (`01_retizero_data.ipynb`, not built yet).
- **Paper tables.** These runs are summarised here. They are not yet read by `finetuning/paper_tables/build_paper_tables.py`, and the paired comparison with Qwen on the 361 images is not built.
- **Exporting a new checkpoint in the app's format**, and Grad-CAM. Both are open questions in the plan.